In [1]:
import os
import pandas as pd
import numpy as np
#from libtiff import TIFF
from matplotlib import pyplot as plt
from skimage.segmentation import find_boundaries
import seaborn as sns
import colorcet as cc
import tifffile
import scipy as sc

### directly get through the all excel by pandas

In [2]:
# reload the all populations
# read in protein CN clustering result

path = '../data/Pro_normed_counts&meta_final.csv'
#Mind that the data we have here are 
df_all_anno = pd.read_csv(path)
df_all_anno.phenoShort.value_counts()

/var/folders/3_/pf5ff_p57fqfsm6w00s_hg480000gn/T/ipykernel_91785/959552837.py:7: DtypeWarning: Columns (79,92,95,96,98,99,100,101,106,108,109,114,115,130,131,132,133,134,135,136,140,141,143,144,146,147,148,150,151,152,154,155,157,158,163,164,165,166,167,168,169,170) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all_anno = pd.read_csv(path)


phenoShort
Carcinoma                                      42022
Spindle carcinoma vs sarcoma                   26022
Sarcoma                                        17469
Benign stroma                                  16892
Myometrium                                     15212
CD4                                             5880
CD8                                             5701
M2                                              5687
DC                                              5228
Perivascular smooth muscle                      4867
Endothelial                                     4788
Other                                           4787
Rhabdomyosarcoma                                3962
HG sarcoma vs PD carcinoma                      2947
M1                                              2268
Benign stroma vs perivascular smooth muscle     1950
B                                               1653
Neutrophil                                       710
Fibroblast desmoplastic stroma popu

In [4]:
# use this to avoid unique panel for each FOV
df_all_anno['phenoShort'] = df_all_anno['phenoShort'].astype('category')

class_names = [
    "Benign stroma vs perivascular smooth muscle",
    "Benign stroma",
    "Endothelial",
    'Fibroblast desmoplastic stroma population',
    'Perivascular smooth muscle',
    "Myometrium",
    "DC",
    'M1',
    'M2',
    "CD4",
    'CD8',
    "B",
    'Neutrophil',
    'Other',
    "HG sarcoma vs PD carcinoma",
    'Rhabdomyosarcoma',
    'Spindle carcinoma vs sarcoma',
    "Sarcoma",
    "Carcinoma"]

# make color panel with more than 20 colors
class_colors = sns.color_palette(['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231', '#911eb4',
                                   '#46f0f0', '#f032e6', '#bcf60c', '#fabebe', '#008080', '#e6beff',
                                    '#9a6324', '#fffac8', '#8298e5', '#5875dc', '#2e51d4', '#ffd8b1',
                                    '#000075', '#808080'])
class_colors

[(0.9019607843137255, 0.09803921568627451, 0.29411764705882354),
 (0.23529411764705882, 0.7058823529411765, 0.29411764705882354),
 (1.0, 0.8823529411764706, 0.09803921568627451),
 (0.2627450980392157, 0.38823529411764707, 0.8470588235294118),
 (0.9607843137254902, 0.5098039215686274, 0.19215686274509805),
 (0.5686274509803921, 0.11764705882352941, 0.7058823529411765),
 (0.27450980392156865, 0.9411764705882353, 0.9411764705882353),
 (0.9411764705882353, 0.19607843137254902, 0.9019607843137255),
 (0.7372549019607844, 0.9647058823529412, 0.047058823529411764),
 (0.9803921568627451, 0.7450980392156863, 0.7450980392156863),
 (0.0, 0.5019607843137255, 0.5019607843137255),
 (0.9019607843137255, 0.7450980392156863, 1.0),
 (0.6039215686274509, 0.38823529411764707, 0.1411764705882353),
 (1.0, 0.9803921568627451, 0.7843137254901961),
 (0.5098039215686274, 0.596078431372549, 0.8980392156862745),
 (0.34509803921568627, 0.4588235294117647, 0.8627450980392157),
 (0.1803921568627451, 0.3176470588235294, 0.8313725490196079),
 (1.0, 0.8470588235294118, 0.6941176470588235),
 (0.0, 0.0, 0.4588235294117647),
 (0.5019607843137255, 0.5019607843137255, 0.5019607843137255)]

In [5]:
df_all_anno['tma_fov'] = df_all_anno['TMA.y'].astype(str) + '_' + df_all_anno['FOV_RNA'].astype(str)
df_all_anno.head()

,cellLabel,X_cent,Y_cent,cellSize,cellWidth,cellHeight,M_CD3,M_CD45,M_CD298.B2M,M_DAPI,...,sarcoma.NOS,rhabdomyosarcoma,LVSI,other,TMA.y,FOV_RNA,FOV_Pro,ann,MS,tma_fov
0,2,1599,11,736,38,22,90.110492,55.763680,471.838796,1067.555044,...,0.0,0,0.0,could be tiny benign epithlium and tiny endome...,4,21,20,A2 Carcinoma,2,4_21
1,3,3242,7,420,38,14,77.780481,52.892032,1401.057743,1355.246933,...,0.0,0,0.0,could be tiny benign epithlium and tiny endome...,4,21,20,A2 Carcinoma,2,4_21
2,5,1780,9,584,36,20,391.613399,111.590114,1725.699503,3476.601227,...,0.0,0,0.0,could be tiny benign epithlium and tiny endome...,4,21,20,A2 Carcinoma,2,4_21
3,6,3022,9,556,38,20,74.173350,50.010344,1217.086549,1293.019536,...,0.0,0,0.0,could be tiny benign epithlium and tiny endome...,4,21,20,A2 Carcinoma,2,4_21
4,8,3712,10,592,32,24,83.255186,48.110137,1580.259593,1009.820872,...,0.0,0,0.0,could be tiny benign epithlium and tiny endome...,4,21,20,A2 Carcinoma,2,4_21


In [18]:

#csvpath3 = '/Users/bokaizhu/Nolan\ Lab\ Dropbox/Zhu\ Bokai/data\ temp/20221117_withNanoString_SMI-0142_BrookeHowitt/5\ Raw\ data/'

fromPath = '../../../../../../Stanford_less_frequent/data temp/20240104_temp_CosMx-Protein_forBrooke/CosMX-SMI_Segmentation_masks/'

for fileName in df_all_anno['tma_fov'].unique():#df_all_anno['fileLabel'].unique():
    # plot specific FOV
    tempDf = df_all_anno[df_all_anno['tma_fov'] == fileName]
    print('Generating prediction maps for image: %s' % fileName)
    
    # load cell boundaries
    currTMA = fileName.split('_')[0]
    currFOV = fileName.split('_')[-1]
    currFOV_pro = np.unique(tempDf['FOV_Pro'])[0].astype(str)

    if currTMA == '4':
        subpath = 'S1_TMA4_seg/'
    if currTMA == '7':
        subpath = 'S3_TMA7_seg/'

    if int(currFOV_pro) < 10:
        cellBoundary_path = fromPath + subpath+'F00'+currFOV_pro+'/0.25mpp_0.075maxima_0.05interior/seg_outline.tiff'
        cell_ins_map_path = fromPath + subpath+'F00'+currFOV_pro+'/0.25mpp_0.075maxima_0.05interior/MESMER_mask.tiff'
        #cellBoundary
    else:
        cellBoundary_path = fromPath + subpath+'F0'+currFOV_pro+'/0.25mpp_0.075maxima_0.05interior/seg_outline.tiff'
        cell_ins_map_path = fromPath + subpath+'F0'+currFOV_pro+'/0.25mpp_0.075maxima_0.05interior/MESMER_mask.tiff'
    # get the cell boundaries
    cellBoundary = tifffile.imread(cellBoundary_path)
    cellBoundary.astype('uint8')
    # dilate the boundary to make the boundaries more obvious
    struct1 = sc.ndimage.generate_binary_structure(2, 1)
    cellBoundary = sc.ndimage.binary_dilation(cellBoundary, structure=struct1,iterations=3).astype(cellBoundary.dtype)
    
    # get the seg
    cell_ins_map = tifffile.imread(cell_ins_map_path)

    r = np.zeros((cellBoundary.shape[0], cellBoundary.shape[1]))
    g = np.zeros((cellBoundary.shape[0], cellBoundary.shape[1]))
    b = np.zeros((cellBoundary.shape[0], cellBoundary.shape[1]))

    cell_count = tempDf.shape[0]
    for i, row in tempDf.iterrows():
        print('Cell %d/%d - %s         ' % (int(row['cellLabel']), cell_count, fileName), end='\r')
        
        cell_id = int(row['cellLabel'])
        cell_pred = row['phenoShort']
        
        mask = cell_ins_map == cell_id
        r[mask] = class_colors[class_names.index(cell_pred)][0]
        g[mask] = class_colors[class_names.index(cell_pred)][1]
        b[mask] = class_colors[class_names.index(cell_pred)][2]

    print('')
    rgb = np.stack([r, g, b], axis=-1)
    # draw the boundaries
    rgb[:,:,0] = rgb[:,:,0]*255 + 40*cellBoundary
    rgb[:,:,1] = rgb[:,:,1]*255 + 40*cellBoundary
    rgb[:,:,2] = rgb[:,:,2]*255 + 40*cellBoundary

    rgb[rgb>255] = 255
    
    # define results dir
    resultDir = os.path.join('../plot/pro_pheno_low/')
    plt.imsave(os.path.join(resultDir, fileName+'.png'), np.uint8(rgb))

Generating prediction maps for image: 4_21
Cell 4869/4288 - 4_21         
Generating prediction maps for image: 4_17
Cell 4936/4682 - 4_17         
Generating prediction maps for image: 4_12
Cell 3483/3020 - 4_12         
Generating prediction maps for image: 4_19
Cell 4048/3526 - 4_19         
Generating prediction maps for image: 4_6
Cell 3170/2419 - 4_6         
Generating prediction maps for image: 7_3
Cell 4833/4155 - 7_3         
Generating prediction maps for image: 7_11
Cell 5667/4850 - 7_11         
Generating prediction maps for image: 7_12
Cell 5326/4171 - 7_12         
Generating prediction maps for image: 4_1
Cell 4851/4214 - 4_1         
Generating prediction maps for image: 4_2
Cell 5418/4690 - 4_2         
Generating prediction maps for image: 4_4
Cell 5819/4358 - 4_4         
Generating prediction maps for image: 4_8
